In [ ]:
%%time

import pandas as pd
import re
import torch
from transformers import pipeline, AutoTokenizer, logging
logging.set_verbosity_error()

# ---------- 0. Configurar dispositivo ----------

device = 0 if torch.cuda.is_available() else -1  # el pipeline de transformers usa 0/-1, no "cuda"/"cpu"
print(f"Usando GPU: {torch.cuda.is_available()}")

# ---------- 1. Cargar el dataset y excluir películas no-inglesas ----------

df = pd.read_pickle('Dataset_final_NLP.pkl')

# peliculas_excluidas = ['Talk to Her', 'Anatomy of a Fall']
# df = df[~df['Title'].isin(peliculas_excluidas)].reset_index(drop=True)

# NOTA: aquí NO deduplicamos por IMDb_ID, porque queremos mantener
# una fila por cada nominación (Award) tal como acordamos para este dataset

# ---------- 2. Configurar el modelo de emociones de Hartmann ----------

MODEL_NAME = "j-hartmann/emotion-english-distilroberta-base"
MAX_TOKENS = 512

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
emotion_analyzer = pipeline(
    "text-classification",
    model=MODEL_NAME,
    tokenizer=tokenizer,
    top_k=None,  # devuelve las probabilidades de las 7 emociones, no solo la ganadora
    device=device
)

EMOCIONES = ['anger', 'disgust', 'fear', 'joy', 'neutral', 'sadness', 'surprise']

# ---------- 3. Chunking por oraciones (mismo sistema que con DistilBERT) ----------

def split_sentences(texto):
    oraciones = re.split(r'(?<=[.!?])\s+', texto.strip())
    return [o for o in oraciones if o.strip() != ""]

def chunk_by_sentences(texto, max_tokens=MAX_TOKENS - 2):
    oraciones = split_sentences(texto)
    chunks = []
    chunk_actual, tokens_actual = [], 0

    for oracion in oraciones:
        n_tokens = len(tokenizer.encode(oracion, add_special_tokens=False))

        if n_tokens > max_tokens:
            if chunk_actual:
                chunks.append(" ".join(chunk_actual))
                chunk_actual, tokens_actual = [], 0
            chunks.append(oracion)
            continue

        if tokens_actual + n_tokens > max_tokens:
            chunks.append(" ".join(chunk_actual))
            chunk_actual, tokens_actual = [oracion], n_tokens
        else:
            chunk_actual.append(oracion)
            tokens_actual += n_tokens

    if chunk_actual:
        chunks.append(" ".join(chunk_actual))

    return chunks

# ---------- 4. Análisis de emociones con media ponderada por tokens ----------

def analyze_emotions(texto):
    if not texto or not isinstance(texto, str) or texto.strip() == "":
        return {e: None for e in EMOCIONES}, None, 0

    chunks = chunk_by_sentences(texto)
    salidas = emotion_analyzer(chunks, batch_size=8, truncation=True)

    # salidas es una lista (por chunk) de listas de dicts [{'label':.., 'score':..}, ...] para las 7 emociones
    emotion_sums = {e: 0.0 for e in EMOCIONES}
    total_tokens = 0

    for chunk, salida_chunk in zip(chunks, salidas):
        n_tokens = len(tokenizer.encode(chunk, add_special_tokens=False))
        for item in salida_chunk:
            emotion_sums[item['label']] += item['score'] * n_tokens
        total_tokens += n_tokens

    emotion_avg = {e: round(v / total_tokens, 4) for e, v in emotion_sums.items()}
    emotion_dominant = max(emotion_avg, key=emotion_avg.get)

    return emotion_avg, emotion_dominant, len(chunks)

# ---------- 5. Aplicar a todo el dataset, ya en formato aplanado ----------

filas = []
for idx, row in df.iterrows():
    script_dict = row['Script_Dict']
    genders_dict = row.get('Characters_Genders', {})

    for personaje, texto in script_dict.items():
        if not texto or not isinstance(texto, str) or texto.strip() == "":
            continue

        emotion_avg, emotion_dominant, n_chunks = analyze_emotions(texto)

        fila = {
            'IMDb_ID': row['IMDb_ID'],
            'Title': row['Title'],
            'Award': row['Award'],
            'Oscar_Year': row['Oscar_Year'],
            'Character': personaje,
            'Gender': genders_dict.get(personaje, 'unknown'),
            'Words': len(texto.split()),
            'N_Chunks': n_chunks,
            'Emotion_Dominant': emotion_dominant
        }
        for e in EMOCIONES:
            fila[f'Emotion_{e.capitalize()}'] = emotion_avg[e]

        filas.append(fila)

df_hartmann = pd.DataFrame(filas)

# ---------- 6. Guardar resultados ----------

df_hartmann.to_pickle('df_hartmann_emotions.pkl')
df_hartmann.to_csv('df_hartmann_emotions.csv', index=False)

print(df_hartmann.head())

c:\Users\gleds\venvs\ProyectoCine_venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Usando GPU: True


Loading weights: 100%|██████████| 105/105 [00:00<00:00, 26298.93it/s]


     IMDb_ID                                          Title         Award  \
0  tt0167260  The Lord of the Rings: The Return of the King  Best Picture   
1  tt0167260  The Lord of the Rings: The Return of the King  Best Picture   
2  tt0167260  The Lord of the Rings: The Return of the King  Best Picture   
3  tt0167260  The Lord of the Rings: The Return of the King  Best Picture   
4  tt0167260  The Lord of the Rings: The Return of the King  Best Picture   

   Oscar_Year Character  Gender  Words  N_Chunks Emotion_Dominant  \
0        2004   ARAGORN    male    493         2             fear   
1        2004     ARWEN  female    127         1          sadness   
2        2004     BILBO    male     54         1          sadness   
3        2004  DENETHOR    male    351         1            anger   
4        2004    DÉAGOL    male      9         1         surprise   

   Emotion_Anger  Emotion_Disgust  Emotion_Fear  Emotion_Joy  Emotion_Neutral  \
0         0.2097           0.0094        

In [6]:

print("Filas totales:", len(df_hartmann))
print("Películas únicas (IMDb_ID):", df_hartmann['IMDb_ID'].nunique())
print("Aragorn aparece", (df_hartmann['Character']=='ARAGORN').sum(), "veces")

Filas totales: 3119
Películas únicas (IMDb_ID): 56
Aragorn aparece 2 veces


In [7]:
df_hartmann.to_pickle('df_hartmann_emotions.pkl')
df_hartmann.to_csv('df_hartmann_emotions.csv', index=False)